# Edge-field diagnostics — what's inside `weighted_edge` and what's wrong

## Scope

This notebook documents the diagnostic work that led to **FIND-003** (positive-clipped gradient) and identified **Bug 2** (anisotropic-diffusion rescaling — *tabled*) in `pipelines/V0/lib.edge_global_field`. Done 2026-05-24 on the production test doublet **DB_top100_031** (Fib × Mel).

## V0 pipeline architecture (where this sits)

```
NB 1 (V0_1_18S_coloring):     transcripts + 18S       →  π_abst, confidence, N_eff
NB 2 (V0_2_boundary_estim):   π_abst                  →  binary boundary mask
THIS NB (edge diagnostics):   the edge-field internals (e_k, gates, sum)   ← YOU ARE HERE
NB 3 (V0_3_cutting_depth):    boundary mask + 18S    →  cut field  →  Cellpose-SAM
```

## What this notebook contains

Four self-contained diagnostic figures, run on DB_top100_031:

1. **Component breakdown** — all parts that go into `weighted_edge`: per-lineage `e_k`, edge_total, c², cell_evidence, weighted_edge.
2. **Cumulative gate buildup** — walk through `edge_total → · c² → · evidence`, showing boundary mask + histogram at each step.
3. **Bug 1 fix (FIND-003)** — `e_k = ‖∇ max(t_k, 0)‖` instead of `‖∇ tanh(K · s_k)‖`. Kills spurious firings where lineage k never wins.
4. **Bug 2 mechanism** — rescaling by global `2π σ²` over-amplifies isolated anchors in low-conductance pockets. *Tabled this session.*

## Constraints

- Segmentation-agnostic (no CP-SAM mask or doublet table feeds the analysis; the focal mask is overlaid only as eval reference).
- Linear intensity throughout, no log scaling.
- Input shown next to output on every figure.
- Local p5-p95 brightness for spatial display.

## Output

Cached PNGs in `scratch/figures/xenium_skin_db031_*.png`. See `findings_registry.yaml::FIND-003` for the conclusion.

## Visual conventions

- **Yellow contour**: focal cell footprint from existing WSI CP-SAM mask. Eval-only.
- **Lineage palette** (`lib.V0_LINEAGE_COLORS`): Mel=blue · Myeloid=orange · Tcell=green · Plasma=red · **Fib=yellow** · Endo=brown · Kerat=pink.
- **Anchor dots**: coloured by lineage. Black border = current lineage of interest; faded grey border = others.
- **Colored 18S**: per-pixel lineage-weighted blend × 18S brightness, with grayscale fallback at low confidence.
- **Scale bar**: white (on dark) or black (on viridis) — 5–20 µm depending on zoom.
- **Context inset** (top-right corner of every focal-zoom figure): small thumbnail of the full 384-px crop with the focal location marked in yellow.

## Setup — load DB_top100_031, compute posterior and per-lineage chains

All heavy compute happens once here; downstream cells just rebuild figures from these variables. Re-run this cell if anything upstream changes.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
REPO = Path.cwd().parents[0]
sys.path.insert(0, str(REPO / 'pipelines' / 'V0'))
import numpy as np, pandas as pd, tifffile, matplotlib.pyplot as plt
from skimage.segmentation import find_boundaries
from scipy.ndimage import sobel
import lib
print(f'lib loaded: K={lib.K} lineages, ALPHA={lib.ALPHA}, N_MIN={lib.N_MIN}, K_SHARP={lib.K_SHARP}')
print(f'anisotropic params: N_ITER_ANISO={lib.N_ITER_ANISO}, DT_ANISO={lib.DT_ANISO}, sigma_eff_px={(lib.N_ITER_ANISO*lib.DT_ANISO)**0.5:.2f}')

In [ ]:
# --- Compute posterior + per-lineage chains (both original and clipped) ---
BID = 'DB_top100_031'
CROP_PX = 384
ZOOM_PAD = 30
MUM_PER_PX = 0.2125
K_SHARP = lib.K_SHARP
LIN_COLORS = lib.V0_LINEAGE_COLORS
LIN_ARR = np.array(LIN_COLORS[:lib.K], dtype=np.float32)

bench = pd.read_parquet(lib.DATA / 'benchmark_doublets.parquet')
row = bench[bench.benchmark_id == BID].iloc[0]
cx, cy = float(row.x_um), float(row.y_um)
cps_focal = int(row.cps_id_at_bookmark)
lin_a, lin_b = row.lineage_pair.split(' × ')
ka, kb = lib.LIN_TO_IDX[lin_a], lib.LIN_TO_IDX[lin_b]

with tifffile.TiffFile(lib.DAPI_TIF) as tf:
    wsi_H, wsi_W = tf.series[0].shape[-2:]
y0, y1, x0, x1 = lib.make_roi_bbox_centred(cx, cy, CROP_PX, wsi_H, wsi_W)
dapi, s18 = lib.load_morphology(y0, y1, x0, x1)
H, W = s18.shape
py, px, li = lib.load_anchors_in_bbox(y0, y1, x0, x1, lib.load_lineage_label_map())

# Verification (per Scientist standing instruction #2)
print(f'doublet={BID}  pair={lin_a} × {lin_b}  ka={ka}, kb={kb}')
print(f'crop shape: dapi={dapi.shape} {dapi.dtype}  s18={s18.shape} {s18.dtype}')
print(f'anchors: {len(py)} total in crop  (per lineage: {np.bincount(li, minlength=lib.K).tolist()})')

# Reproduce posterior pipeline so we can also see rho_smooth (pre-rescaling)
rho_pts = np.zeros((lib.K, H, W), dtype=np.float32)
np.add.at(rho_pts, (li, py, px), 1.0)
cond = lib.make_conductance(s18, mode='linear')
rho_smooth = lib.diffuse_anisotropic(rho_pts, cond, n_iter=lib.N_ITER_ANISO, dt=lib.DT_ANISO)
sigma_eff_px = (lib.N_ITER_ANISO * lib.DT_ANISO) ** 0.5
scale_factor = 2.0 * np.pi * sigma_eff_px ** 2
N_eff = rho_smooth * scale_factor
N_total = N_eff.sum(axis=0)
pi_post = (N_eff + lib.ALPHA) / (N_total + lib.K * lib.ALPHA + 1e-9)[None]
confidence = N_total / (N_total + lib.N_MIN + 1e-9)
pi_abst = (confidence[None] * pi_post + (1.0 - confidence[None]) * (1.0 / lib.K)).astype(np.float32)
top_idx = np.argmax(pi_abst, axis=0).astype(np.int8)
evidence = lib.cell_evidence(dapi, s18, lib.load_wsi_percentiles())
c2 = (confidence ** 2).astype(np.float32)

# Per-lineage chain (ORIGINAL and CLIPPED variants of e_k)
top1_val = np.take_along_axis(pi_abst, top_idx[None].astype(np.int64), axis=0)[0]
masked = np.where(np.arange(lib.K, dtype=np.int8)[:, None, None] == top_idx[None], -np.inf, pi_abst)
top2_val = np.max(masked, axis=0)

s_all = np.zeros((lib.K, H, W), dtype=np.float32)
t_all = np.zeros((lib.K, H, W), dtype=np.float32)
e_orig = np.zeros((lib.K, H, W), dtype=np.float32)
e_clip = np.zeros((lib.K, H, W), dtype=np.float32)
for k in range(lib.K):
    others = np.where(top_idx == k, top2_val, top1_val)
    s_all[k] = pi_abst[k] - others
    t_all[k] = np.tanh(K_SHARP * s_all[k]).astype(np.float32)
    # original
    gy = sobel(t_all[k], axis=0); gx = sobel(t_all[k], axis=1)
    e_orig[k] = np.hypot(gx, gy).astype(np.float32)
    # clipped (FIND-003)
    tp = np.maximum(t_all[k], 0).astype(np.float32)
    gy = sobel(tp, axis=0); gx = sobel(tp, axis=1)
    e_clip[k] = np.hypot(gx, gy).astype(np.float32)

edge_total_o = e_orig.sum(0)
edge_total_c = e_clip.sum(0)
weighted_edge_o = (edge_total_o * c2 * evidence).astype(np.float32)
weighted_edge_c = (edge_total_c * c2 * evidence).astype(np.float32)

# Focal mask + zoom region
wsi_mask = tifffile.imread(lib.DATA / 'cpsam_whole_slide' / 'masks.tif')
focal = (wsi_mask[y0:y1, x0:x1] == cps_focal); del wsi_mask
ys, xs = np.where(focal)
zy0 = max(int(ys.min())-ZOOM_PAD, 0); zy1 = min(int(ys.max())+ZOOM_PAD, H)
zx0 = max(int(xs.min())-ZOOM_PAD, 0); zx1 = min(int(xs.max())+ZOOM_PAD, W)
zoom = (slice(zy0, zy1), slice(zx0, zx1))
focal_z = focal[zoom]; Hz, Wz = focal_z.shape

# Anchors restricted to zoom (for overlays)
in_z = (py >= zy0) & (py < zy1) & (px >= zx0) & (px < zx1)
py_z = py[in_z] - zy0; px_z = px[in_z] - zx0; li_z = li[in_z]

# Colored 18S (blended)
lo, hi = np.percentile(s18, [5, 95])
s18n = np.clip((s18.astype(np.float32) - lo) / max(hi - lo, 1e-6), 0, 1)
N_tot_eff = N_eff.sum(0)
pi_ml = N_eff / np.maximum(N_tot_eff, 1e-9)[None]
color = np.einsum('khw,kc->hwc', pi_ml, LIN_ARR)
c3 = confidence[..., None].astype(np.float32)
colored = (color * s18n[..., None] * c3 + np.stack([s18n]*3, axis=-1) * (1-c3)).astype(np.float32)

print(f'\nsetup done: focal {focal.sum()} px, zoom {focal_z.shape}, anchors_in_zoom {in_z.sum()}')
print(f'sigma_eff_px = {sigma_eff_px:.2f}, 2πσ² = {scale_factor:.1f}')
print(f'edge_total_orig max in focal: {edge_total_o[focal].max():.2f}; clip: {edge_total_c[focal].max():.2f}')
print(f'weighted_edge_orig max in focal: {weighted_edge_o[focal].max():.2f}; clip: {weighted_edge_c[focal].max():.2f}')

## Helper plotting functions

Shared across all diagnostics: scale bar, focal-cell contour, context inset (full-crop thumbnail with focal location), anchor overlay.

In [ ]:
FIG_DIR = Path('scratch/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

def scalebar(ax, h_local, length_um=20, color='white'):
    L_px = length_um / MUM_PER_PX
    ax.plot([8, 8+L_px], [h_local-12, h_local-12], color=color, lw=3, solid_capstyle='butt')
    ax.text(8 + L_px/2, h_local-18, f'{length_um} µm', color=color,
            ha='center', va='bottom', fontsize=8)

def focal_contour(ax, focal_mask=None):
    fm = focal_mask if focal_mask is not None else focal_z
    fb = find_boundaries(fm, mode='outer').astype(int)
    ax.contour(fb, levels=[0.5], colors='yellow', linewidths=1.4)

def overlay_anchors(ax, size=10, only_lineage=None):
    for k in range(lib.K):
        m = li_z == k
        if not m.any():
            continue
        s = size if (only_lineage is None or k == only_lineage) else 4
        a = 0.85 if (only_lineage is None or k == only_lineage) else 0.35
        ec = 'black' if (only_lineage is None or k == only_lineage) else 'none'
        ax.scatter(px_z[m], py_z[m], s=s, c=[LIN_COLORS[k]],
                   edgecolors=ec, linewidths=0.3, alpha=a)

def setup_panel(ax, title, sbcolor='white', length_um=20):
    focal_contour(ax); scalebar(ax, Hz, length_um=length_um, color=sbcolor)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])

def add_context_inset(ax, frac=0.22):
    """Add a small inset showing the full 384-px crop with the focal location
    marked in yellow and the current zoom area marked in cyan. FIGURE_STANDARDS
    rule: every focal-zoom display needs a context panel."""
    from mpl_toolkits.axes_grid1.inset_locator import inset_axes
    iax = inset_axes(ax, width=f'{int(frac*100)}%', height=f'{int(frac*100)}%',
                     loc='upper right', borderpad=0.3)
    iax.imshow(colored)
    fb_full = find_boundaries(focal, mode='outer').astype(int)
    iax.contour(fb_full, levels=[0.5], colors='yellow', linewidths=0.6)
    # zoom rectangle
    from matplotlib.patches import Rectangle
    iax.add_patch(Rectangle((zx0, zy0), zx1-zx0, zy1-zy0, fill=False,
                            edgecolor='cyan', linewidth=0.8))
    iax.set_xticks([]); iax.set_yticks([])
    for s in iax.spines.values():
        s.set_edgecolor('white'); s.set_linewidth(0.8)
    return iax

print('helpers ready')

## Diagnostic 1 — Component breakdown of `weighted_edge`

All inputs that go into the final scalar field, broken out for inspection. 16 panels:

- Row 1 (4): per-lineage `e_k` for the focal pair + 2 other lineages
- Row 2 (4): remaining 3 lineages + `edge_total = Σ_k e_k`
- Row 3 (4): gates and cumulative product (`c²`, `·c²`, `cell_evidence`, `weighted_edge`)
- Row 4 (4): boundary mask @ p99 + histogram + focal-pair attribution

Uses the CLIPPED formulation per FIND-003.

In [ ]:
# === Diagnostic 1: component breakdown of weighted_edge_clipped ===
fig, axes = plt.subplots(4, 4, figsize=(20, 20))
fig.suptitle(f'All components going into weighted_edge_clipped on {BID} ({lin_a} × {lin_b})\n'
             f'(Σ_k ‖∇ max(t_k, 0)‖) · c² · evidence  |  K_sharp={K_SHARP}',
             fontsize=13, y=0.998)

vmax_e = float(np.percentile(e_clip, 99.9))
vmax_et = float(np.percentile(edge_total_c, 99.5))
vmax_wc2 = float(np.percentile(edge_total_c * c2, 99.5))
vmax_we = float(np.percentile(weighted_edge_c, 99.5))
wedge_c2 = edge_total_c * c2
tau_w = float(np.percentile(weighted_edge_c[weighted_edge_c > 0], 99.0))
boundary = weighted_edge_c > tau_w

# Order: focal pair first, then others, then edge_total in slot 8
lineage_order = [ka, kb] + [k for k in range(lib.K) if k not in (ka, kb)]

# Row 1: focal pair + 2 next lineages
for c_, k in enumerate(lineage_order[:4]):
    ax = axes[0, c_]
    ax.imshow(e_clip[k, zoom[0], zoom[1]], cmap='hot', vmin=0, vmax=vmax_e)
    setup_panel(ax, f'e_{lib.LINEAGES[k]}  (max in focal: {e_clip[k][focal].max():.2f})')
    ax.add_patch(plt.Rectangle((2, 2), 18, 18, color=LIN_COLORS[k], ec='black'))

# Row 2: remaining lineages + edge_total
for c_, k in enumerate(lineage_order[4:]):
    ax = axes[1, c_]
    ax.imshow(e_clip[k, zoom[0], zoom[1]], cmap='hot', vmin=0, vmax=vmax_e)
    setup_panel(ax, f'e_{lib.LINEAGES[k]}  (max in focal: {e_clip[k][focal].max():.2f})')
    ax.add_patch(plt.Rectangle((2, 2), 18, 18, color=LIN_COLORS[k], ec='black'))
ax = axes[1, 3]
ax.imshow(edge_total_c[zoom], cmap='hot', vmin=0, vmax=vmax_et)
setup_panel(ax, f'edge_total = Σ_k e_k  (max in focal: {edge_total_c[focal].max():.2f}, off: {edge_total_c[~focal].max():.2f})')
add_context_inset(ax)

# Row 3: gates and cumulative product
ax = axes[2, 0]; ax.imshow(c2[zoom], cmap='viridis', vmin=0, vmax=1)
setup_panel(ax, f'GATE 1: c² = confidence²  (mean focal: {c2[focal].mean():.2f})', sbcolor='black')
ax = axes[2, 1]; ax.imshow(wedge_c2[zoom], cmap='hot', vmin=0, vmax=vmax_wc2)
setup_panel(ax, f'edge_total · c²  (max focal: {wedge_c2[focal].max():.2f}, off: {wedge_c2[~focal].max():.2f})')
ax = axes[2, 2]; ax.imshow(evidence[zoom], cmap='viridis', vmin=0, vmax=1)
setup_panel(ax, f'GATE 2: cell_evidence (mean focal: {evidence[focal].mean():.2f})', sbcolor='black')
ax = axes[2, 3]; ax.imshow(weighted_edge_c[zoom], cmap='hot', vmin=0, vmax=vmax_we)
setup_panel(ax, f'weighted_edge = edge_total · c² · evidence  (max focal: {weighted_edge_c[focal].max():.2f})')

# Row 4: boundary inspection
ax = axes[3, 0]; ax.imshow(weighted_edge_c[zoom], cmap='hot', vmin=0, vmax=vmax_we)
ax.contour(boundary[zoom], levels=[0.5], colors='cyan', linewidths=1.2)
setup_panel(ax, f'weighted_edge + p99 contour (τ_w={tau_w:.2f}, GLOBAL)')

ax = axes[3, 1]; ax.imshow(colored[zoom])
b_z = boundary[zoom]
ov = np.zeros((Hz, Wz, 4), dtype=np.float32); ov[b_z] = (0.0, 1.0, 1.0, 0.75)
ax.imshow(ov, interpolation='none'); overlay_anchors(ax)
setup_panel(ax, f'boundary @ p99 on colored 18S  (n_bnd_in_focal={(boundary & focal).sum()})')

ax = axes[3, 2]
we_f = weighted_edge_c[focal]; we_o = weighted_edge_c[~focal]
bins = np.linspace(0, vmax_we * 1.2, 60)
ax.hist(we_o, bins=bins, alpha=0.5, label=f'off-focal (n={len(we_o):,})', color='gray')
ax.hist(we_f, bins=bins, alpha=0.75, label=f'in focal (n={len(we_f):,})', color='C1')
ax.axvline(tau_w, color='red', linestyle='--', linewidth=2, label=f'p99 = {tau_w:.2f}')
ax.set_xlabel('weighted_edge value'); ax.set_ylabel('# pixels')
ax.set_title('weighted_edge: focal vs off-focal\n(focal mass below global p99)', fontsize=10)
ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[3, 3]
attribution = (e_clip[ka] + e_clip[kb]) / np.maximum(edge_total_c, 1e-9)
ax.imshow(attribution[zoom], cmap='viridis', vmin=0, vmax=1)
setup_panel(ax, f'(e_{lib.LINEAGES[ka]} + e_{lib.LINEAGES[kb]}) / edge_total\n(focal-pair share; 1=pure)', sbcolor='black')

plt.tight_layout(rect=[0, 0, 1, 0.985])
out = FIG_DIR / 'xenium_skin_db031_weighted-edge-clipped-components_K-sharp8.png'
plt.savefig(out, dpi=130, bbox_inches='tight'); plt.show()
print(f'saved: {out}')

## Diagnostic 2 — Cumulative gate buildup

Walk through `edge_total → · c² → · evidence`, redrawing boundary mask + histogram at each step. Shows whether each gate moves the boundary INTO or AWAY FROM the focal cell.

Spoiler from the table below: gates don't move `n_bnd_in_focal` off zero. The fundamental issue is off-focal dominance, not the gates themselves.

In [ ]:
# === Diagnostic 2: cumulative buildup ===
F0 = edge_total_c.astype(np.float32)
F1 = (edge_total_c * c2).astype(np.float32)
F2 = weighted_edge_c.astype(np.float32)

def boundary_at_p99(F):
    nz = F[F > 0]
    if not nz.size: return F > np.inf, 0.0
    t = float(np.percentile(nz, 99.0))
    return F > t, t
b0, t0_ = boundary_at_p99(F0)
b1, t1_ = boundary_at_p99(F1)
b2, t2_ = boundary_at_p99(F2)

vmax_F = float(np.percentile(F0, 99.5))

fig, axes = plt.subplots(3, 4, figsize=(20, 15))
fig.suptitle(f'V0 edge-field CUMULATIVE buildup on {BID} ({lin_a} × {lin_b})\n'
             'each row adds one gate; col 3 = boundary @ p99 at that step', fontsize=13, y=0.998)

def hist_panel(ax, F, tau, label):
    bins = np.linspace(0, vmax_F * 1.05, 60)
    ax.hist(F[~focal], bins=bins, alpha=0.5, label=f'off-focal (n={(~focal).sum():,})', color='gray')
    ax.hist(F[focal], bins=bins, alpha=0.75, label=f'in focal (n={focal.sum():,})', color='C1')
    ax.axvline(tau, color='red', linestyle='--', linewidth=2, label=f'p99 = {tau:.2f}')
    ia = int((F[focal] > tau).sum()); oa = int((F[~focal] > tau).sum())
    ax.set_xlabel(f'{label} value'); ax.set_ylabel('# px')
    ax.set_title(f'{label}: in_focal>τ={ia}  off-focal>τ={oa}', fontsize=10)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Row 1: start with edge_total
ax = axes[0,0]; ax.imshow(colored[zoom]); overlay_anchors(ax)
setup_panel(ax, 'INPUT (no gate yet)\ncolored 18S + anchors')
add_context_inset(ax)
ax = axes[0,1]; ax.imshow(F0[zoom], cmap='hot', vmin=0, vmax=vmax_F)
setup_panel(ax, f'STEP 0: edge_total = Σ_k ‖∇ max(t_k,0)‖\nmax focal: {F0[focal].max():.2f}  off: {F0[~focal].max():.2f}')
ax = axes[0,2]; ax.imshow(colored[zoom])
b0z = b0[zoom]; ov = np.zeros((Hz, Wz, 4), dtype=np.float32); ov[b0z] = (0.0, 1.0, 1.0, 0.75)
ax.imshow(ov, interpolation='none'); overlay_anchors(ax)
setup_panel(ax, f'boundary @ p99(edge_total), τ={t0_:.2f}\nn_bnd_in_focal = {(b0 & focal).sum()}')
hist_panel(axes[0,3], F0, t0_, 'edge_total')

# Row 2: + c²
ax = axes[1,0]; ax.imshow(c2[zoom], cmap='viridis', vmin=0, vmax=1)
setup_panel(ax, f'GATE 1: c²  (mean focal: {c2[focal].mean():.2f}, off: {c2[~focal].mean():.2f})', sbcolor='black')
ax = axes[1,1]; ax.imshow(F1[zoom], cmap='hot', vmin=0, vmax=vmax_F)
setup_panel(ax, f'STEP 1: edge_total · c²\nmax focal: {F1[focal].max():.2f}  off: {F1[~focal].max():.2f}')
ax = axes[1,2]; ax.imshow(colored[zoom])
b1z = b1[zoom]; ov = np.zeros((Hz, Wz, 4), dtype=np.float32); ov[b1z] = (0.0, 1.0, 1.0, 0.75)
ax.imshow(ov, interpolation='none'); overlay_anchors(ax)
setup_panel(ax, f'boundary @ p99(edge_total · c²), τ={t1_:.2f}\nn_bnd_in_focal = {(b1 & focal).sum()}')
hist_panel(axes[1,3], F1, t1_, 'edge_total · c²')

# Row 3: + evidence
ax = axes[2,0]; ax.imshow(evidence[zoom], cmap='viridis', vmin=0, vmax=1)
setup_panel(ax, f'GATE 2: cell_evidence  (mean focal: {evidence[focal].mean():.2f}, off: {evidence[~focal].mean():.2f})', sbcolor='black')
ax = axes[2,1]; ax.imshow(F2[zoom], cmap='hot', vmin=0, vmax=vmax_F)
setup_panel(ax, f'STEP 2: · c² · evidence (= weighted_edge)\nmax focal: {F2[focal].max():.2f}  off: {F2[~focal].max():.2f}')
ax = axes[2,2]; ax.imshow(colored[zoom])
b2z = b2[zoom]; ov = np.zeros((Hz, Wz, 4), dtype=np.float32); ov[b2z] = (0.0, 1.0, 1.0, 0.75)
ax.imshow(ov, interpolation='none'); overlay_anchors(ax)
setup_panel(ax, f'boundary @ p99(weighted_edge), τ={t2_:.2f}\nn_bnd_in_focal = {(b2 & focal).sum()}')
hist_panel(axes[2,3], F2, t2_, 'weighted_edge')

plt.tight_layout(rect=[0, 0, 1, 0.985])
out = FIG_DIR / 'xenium_skin_db031_edge-field-cumulative_anisotropic-sigma2_K-sharp8.png'
plt.savefig(out, dpi=130, bbox_inches='tight'); plt.show()
print(f'saved: {out}')

print('\nCumulative summary table:')
print(f"{'Stage':45s} | {'τ@p99':>7s} | {'max focal':>10s} | {'max off':>10s} | {'ratio':>7s} | {'n_in_focal':>11s} | {'n_off':>7s}")
for L, F, t, b in [('Step 0: edge_total (no gate)', F0, t0_, b0),
                    ('Step 1: × c²', F1, t1_, b1),
                    ('Step 2: × c² · evidence', F2, t2_, b2)]:
    mf, mo = F[focal].max(), F[~focal].max()
    print(f'{L:45s} | {t:7.3f} | {mf:10.3f} | {mo:10.3f} | {mo/max(mf,1e-9):6.2f}x | {(b&focal).sum():>11d} | {(b&~focal).sum():>7d}')

## Diagnostic 3 — Bug 1 fix: positive-clipping the gradient (FIND-003)

Original: `e_k = ‖∇ tanh(K_sharp · s_k)‖` fires on every zero-crossing of `s_k`, including transitions where lineage k *never wins on either side* (s_k flips from "slightly losing" to "strongly losing").

Fix: `e_k = ‖∇ max(t_k, 0)‖`. Clipping to the positive part means the gradient is non-zero only at the perimeter of the lineage-k-winning region. Structurally correct.

**Verified on this doublet (8,174 bright clipped-e_Fib pixels):** every single one has a positive `t_Fib` neighbor within 3×3. Zero false positives. See `findings_registry.yaml::FIND-003` for the full registry entry.

The figure compares before/after at (a) a spurious region with no nearby Fib anchors, and (b) the focal cell's real Fib boundary.

In [ ]:
# === Diagnostic 3: Bug 1 fix (FIND-003) ===
from scipy.ndimage import distance_transform_edt

# Locate a spurious pixel: bright e_Fib_orig, t_Fib < 0, no Fib anchor within 20 px
fib_mask_pts = np.zeros((H, W), dtype=bool)
fib_py_arr = py[li == ka]; fib_px_arr = px[li == ka]
fib_mask_pts[fib_py_arr, fib_px_arr] = True
dist_fib = distance_transform_edt(~fib_mask_pts)
spurious = (e_orig[ka] > 1.0) & (dist_fib > 20) & (t_all[ka] <= 0)
ys_s, xs_s = np.where(spurious)
idx = np.argmax(e_orig[ka][spurious])
y_sp, x_sp = int(ys_s[idx]), int(xs_s[idx])
print(f'Spurious pixel: (y={y_sp}, x={x_sp})')
print(f'  e_Fib orig = {e_orig[ka, y_sp, x_sp]:.2f}, clip = {e_clip[ka, y_sp, x_sp]:.2f}')
print(f'  t_Fib = {t_all[ka, y_sp, x_sp]:.3f},  distance to nearest Fib anchor = {dist_fib[y_sp, x_sp]:.1f} px')

wedge_o_p99 = float(np.percentile(weighted_edge_o[weighted_edge_o > 0], 99.0))
wedge_c_p99 = float(np.percentile(weighted_edge_c[weighted_edge_c > 0], 99.0))
b_o = weighted_edge_o > wedge_o_p99
b_c = weighted_edge_c > wedge_c_p99

fig, axes = plt.subplots(3, 4, figsize=(20, 15))
fig.suptitle('Bug-1 fix (FIND-003): clip t_k to [0, 1] before gradient → e_k = ‖∇ max(t_k, 0)‖\n'
             f'{BID} ({lin_a} × {lin_b}); K_sharp={K_SHARP}', fontsize=13, y=0.998)

vmax_e = float(np.percentile(np.concatenate([e_orig[ka].ravel(), e_clip[ka].ravel()]), 99.5))
vmax_we = float(np.percentile(weighted_edge_o, 99.5))

# Row 1: spurious region (no Fib within 20 px)
ZR = 25
ya, yb_ = max(0, y_sp-ZR), min(H, y_sp+ZR+1)
xa, xb_ = max(0, x_sp-ZR), min(W, x_sp+ZR+1)
hloc = yb_ - ya
yc, xc = y_sp - ya, x_sp - xa

ax = axes[0,0]; ax.imshow(colored[ya:yb_, xa:xb_])
inw = (py >= ya) & (py < yb_) & (px >= xa) & (px < xb_)
for k in range(lib.K):
    m = inw & (li == k)
    if m.any():
        ax.scatter(px[m]-xa, py[m]-ya, s=20, c=[LIN_COLORS[k]],
                   edgecolors='black', linewidths=0.4, alpha=0.95)
ax.plot(xc, yc, 'x', color='lime', markersize=14, markeredgewidth=3)
scalebar(ax, hloc, length_um=5)
ax.set_title(f'SPURIOUS region (no Fib within 20 px)\ncolored 18S + anchors', fontsize=10)
ax.set_xticks([]); ax.set_yticks([])
add_context_inset(ax)

ax = axes[0,1]; im = ax.imshow(t_all[ka, ya:yb_, xa:xb_], cmap='RdBu_r', vmin=-1, vmax=1)
ax.plot(xc, yc, 'x', color='lime', markersize=14, markeredgewidth=3)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
scalebar(ax, hloc, length_um=5)
ax.set_title(f't_Fib (original)\nat marker: {t_all[ka, y_sp, x_sp]:+.3f}', fontsize=10)
ax.set_xticks([]); ax.set_yticks([])

ax = axes[0,2]; im = ax.imshow(e_orig[ka, ya:yb_, xa:xb_], cmap='hot', vmin=0, vmax=vmax_e)
ax.plot(xc, yc, 'x', color='cyan', markersize=14, markeredgewidth=3)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
scalebar(ax, hloc, length_um=5)
ax.set_title(f'e_Fib ORIGINAL = ‖∇ t_Fib‖\nat marker: {e_orig[ka, y_sp, x_sp]:.2f}', fontsize=10)
ax.set_xticks([]); ax.set_yticks([])

ax = axes[0,3]; im = ax.imshow(e_clip[ka, ya:yb_, xa:xb_], cmap='hot', vmin=0, vmax=vmax_e)
ax.plot(xc, yc, 'x', color='cyan', markersize=14, markeredgewidth=3)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
scalebar(ax, hloc, length_um=5)
ax.set_title(f'e_Fib CLIPPED = ‖∇ max(t_Fib, 0)‖\nat marker: {e_clip[ka, y_sp, x_sp]:.2f}', fontsize=10)
ax.set_xticks([]); ax.set_yticks([])

# Row 2: focal cell zoom (real Fib edge)
ax = axes[1,0]; ax.imshow(colored[zoom]); overlay_anchors(ax, size=12)
setup_panel(ax, 'FOCAL CELL (real Fib-Mel boundary)\ncolored 18S + anchors')
add_context_inset(ax)
ax = axes[1,1]; im = ax.imshow(t_all[ka, zoom[0], zoom[1]], cmap='RdBu_r', vmin=-1, vmax=1)
focal_contour(ax); scalebar(ax, Hz)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title('t_Fib (original)', fontsize=10); ax.set_xticks([]); ax.set_yticks([])
ax = axes[1,2]; im = ax.imshow(e_orig[ka, zoom[0], zoom[1]], cmap='hot', vmin=0, vmax=vmax_e)
focal_contour(ax); scalebar(ax, Hz)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title(f'e_Fib ORIGINAL  (max in focal: {e_orig[ka][focal].max():.2f})', fontsize=10)
ax.set_xticks([]); ax.set_yticks([])
ax = axes[1,3]; im = ax.imshow(e_clip[ka, zoom[0], zoom[1]], cmap='hot', vmin=0, vmax=vmax_e)
focal_contour(ax); scalebar(ax, Hz)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title(f'e_Fib CLIPPED  (max in focal: {e_clip[ka][focal].max():.2f})', fontsize=10)
ax.set_xticks([]); ax.set_yticks([])

# Row 3: global aggregation impact
ax = axes[2,0]; ax.imshow(weighted_edge_o[zoom], cmap='hot', vmin=0, vmax=vmax_we)
setup_panel(ax, f'weighted_edge ORIGINAL\nmax in focal: {weighted_edge_o[focal].max():.2f}')
ax = axes[2,1]; ax.imshow(weighted_edge_c[zoom], cmap='hot', vmin=0, vmax=vmax_we)
setup_panel(ax, f'weighted_edge CLIPPED (FIND-003)\nmax in focal: {weighted_edge_c[focal].max():.2f}')

ax = axes[2,2]; ax.imshow(colored[zoom])
boz = b_o[zoom]; ov = np.zeros((Hz, Wz, 4), dtype=np.float32); ov[boz] = (0.0, 1.0, 1.0, 0.75)
ax.imshow(ov, interpolation='none')
setup_panel(ax, f'boundary @ p99 ORIGINAL  n_bnd_in_focal = {(b_o & focal).sum()}')
ax = axes[2,3]; ax.imshow(colored[zoom])
bcz = b_c[zoom]; ov = np.zeros((Hz, Wz, 4), dtype=np.float32); ov[bcz] = (0.0, 1.0, 1.0, 0.75)
ax.imshow(ov, interpolation='none')
setup_panel(ax, f'boundary @ p99 CLIPPED  n_bnd_in_focal = {(b_c & focal).sum()}')

plt.tight_layout(rect=[0, 0, 1, 0.985])
out = FIG_DIR / 'xenium_skin_db031_bug1-fix_relu-clip_K-sharp8.png'
plt.savefig(out, dpi=130, bbox_inches='tight'); plt.show()
print(f'saved: {out}')

# Sanity check (FIND-003 evidence)
from scipy.ndimage import maximum_filter
t_pos = np.maximum(t_all[ka], 0)
t_pos_max = maximum_filter(t_pos, size=3)
mask = e_clip[ka] > 0.5
n_bright = int(mask.sum())
n_no_pos = int((mask & (t_pos_max == 0)).sum())
print(f'\nSanity: {n_bright:,} bright clipped pixels (e_Fib_clip > 0.5);')
print(f'        {n_no_pos:,} of them with NO t_Fib > 0 within 3×3 window (should be 0)')
assert n_no_pos == 0, 'FIND-003 sanity check failed!'

## Diagnostic 4 — Bug 2 mechanism: rescaling by global 2πσ² (TABLED)

**Status: tabled this session.** Captured here for reference and future revisit.

The line `N_eff = rho_smooth * 2 * np.pi * sigma_eff_px ** 2` in `lib.lineage_posterior_anisotropic` assumes each anchor spread to a Gaussian of width σ. In low-conductance pockets where anisotropic diffusion can't spread, the actual spread area is much smaller than `2πσ²` and the rescaling over-amplifies. Side effect: confidence formula `c = N_total / (N_total + N_min)` thinks an over-amplified pocket has high evidence, when in reality only one or a few anchors contributed.

Candidate fix (not implemented): replace global rescaling with local window-integration of `rho_smooth`. See session transcript / planned future work.

In [ ]:
# === Diagnostic 4: Bug 2 mechanism (rescaling factor) ===
# Find an isolated Fib anchor for inspection
fib_py_arr2 = py[li == ka]; fib_px_arr2 = px[li == ka]
fib_yx = np.column_stack([fib_py_arr2, fib_px_arr2])
isolation = []
for i, (yi, xi) in enumerate(fib_yx):
    d2 = (fib_yx[:,0] - yi)**2 + (fib_yx[:,1] - xi)**2
    n_within_15 = int(((d2 > 0) & (d2 < 15**2)).sum())
    isolation.append((n_within_15, i, int(yi), int(xi)))
isolation.sort()
isolated = [it for it in isolation if it[0] == 0]
if isolated:
    # Pick the one with highest local e_Fib
    best = max(isolated, key=lambda it: e_orig[ka, max(0,it[2]-5):it[2]+6, max(0,it[3]-5):it[3]+6].max())
    _, _, y_iso, x_iso = best
else:
    y_iso, x_iso = 40, 2
print(f'Isolated Fib anchor: ({y_iso}, {x_iso})  (no other Fib within 15 px)')
print(f'  conductance c = {cond[y_iso, x_iso]:.3f}')
print(f'  rho_smooth[Fib] = {rho_smooth[ka, y_iso, x_iso]:.4f}')
print(f'  rescaling factor 2πσ² = {scale_factor:.1f}')
print(f'  N_eff[Fib] = rho_smooth × 2πσ² = {N_eff[ka, y_iso, x_iso]:.1f}')

# Compare against a well-mixed Fib region inside focal
focal_high = np.argmax(np.where(focal, N_eff[ka], -np.inf))
y_mix, x_mix = np.unravel_index(focal_high, (H, W))
y_mix, x_mix = int(y_mix), int(x_mix)
print(f'\nWell-mixed Fib region inside focal: ({y_mix}, {x_mix})')
print(f'  conductance c = {cond[y_mix, x_mix]:.3f}')
print(f'  rho_smooth[Fib] = {rho_smooth[ka, y_mix, x_mix]:.4f}')
print(f'  N_eff[Fib] = {N_eff[ka, y_mix, x_mix]:.1f}')

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle(f'Bug 2 mechanism (TABLED) — rescaling by global 2πσ² over-amplifies disconnected anchors\n'
             f'Global rescaling: 2π · {sigma_eff_px:.2f}² = {scale_factor:.0f}', fontsize=13, y=0.998)

def small_panel(ax, t, h_loc, sbcol='white', L=2):
    scalebar(ax, h_loc, length_um=L, color=sbcol)
    ax.set_title(t, fontsize=10); ax.set_xticks([]); ax.set_yticks([])

# Row 1: isolated anchor
ZR = 12
ya, yb_ = max(0, y_iso-ZR), min(H, y_iso+ZR+1)
xa, xb_ = max(0, x_iso-ZR), min(W, x_iso+ZR+1)
Hl = yb_ - ya
yc, xc = y_iso - ya, x_iso - xa
ax = axes[0,0]; ax.imshow(s18n[ya:yb_, xa:xb_], cmap='gray', vmin=0, vmax=1)
inw = (py >= ya) & (py < yb_) & (px >= xa) & (px < xb_)
for k in range(lib.K):
    m = inw & (li == k)
    if m.any():
        s_ = 30 if k == ka else 8
        ec = 'black' if k == ka else 'none'
        a = 0.95 if k == ka else 0.4
        ax.scatter(px[m]-xa, py[m]-ya, s=s_, c=[LIN_COLORS[k]],
                   edgecolors=ec, linewidths=0.5, alpha=a)
ax.plot(xc, yc, 'x', color='lime', markersize=14, markeredgewidth=3)
small_panel(ax, 'ISOLATED single Fib anchor (no other Fib within 15 px)\n18S + anchors (Fib highlighted)', Hl)
add_context_inset(ax)
ax = axes[0,1]; im = ax.imshow(cond[ya:yb_, xa:xb_], cmap='viridis', vmin=0, vmax=1)
ax.plot(xc, yc, 'x', color='lime', markersize=14, markeredgewidth=3)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
small_panel(ax, f'conductance c at marker = {cond[y_iso, x_iso]:.3f}\n(low ⇒ mass can\u2019t escape this pixel)', Hl, sbcol='black')
ax = axes[0,2]; im = ax.imshow(rho_smooth[ka, ya:yb_, xa:xb_], cmap='hot')
ax.plot(xc, yc, 'x', color='lime', markersize=14, markeredgewidth=3)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
small_panel(ax, f'rho_smooth[Fib] (pre-rescaling, mass-conserved)\nat marker: {rho_smooth[ka, y_iso, x_iso]:.4f}', Hl)
ax = axes[0,3]; im = ax.imshow(N_eff[ka, ya:yb_, xa:xb_], cmap='hot')
ax.plot(xc, yc, 'x', color='lime', markersize=14, markeredgewidth=3)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
small_panel(ax, f'N_eff[Fib] = rho_smooth × 2πσ²\nat marker: {N_eff[ka, y_iso, x_iso]:.1f}  (inflated, expected ≈1)', Hl)

# Row 2: well-mixed
ZR2 = 30
ya2, yb2 = max(0, y_mix-ZR2), min(H, y_mix+ZR2+1)
xa2, xb2 = max(0, x_mix-ZR2), min(W, x_mix+ZR2+1)
Hl2 = yb2 - ya2
yc2, xc2 = y_mix - ya2, x_mix - xa2
ax = axes[1,0]; ax.imshow(s18n[ya2:yb2, xa2:xb2], cmap='gray', vmin=0, vmax=1)
inw2 = (py >= ya2) & (py < yb2) & (px >= xa2) & (px < xb2)
for k in range(lib.K):
    m = inw2 & (li == k)
    if m.any():
        s_ = 18 if k == ka else 4
        ec = 'black' if k == ka else 'none'
        a = 0.85 if k == ka else 0.3
        ax.scatter(px[m]-xa2, py[m]-ya2, s=s_, c=[LIN_COLORS[k]],
                   edgecolors=ec, linewidths=0.4, alpha=a)
ax.plot(xc2, yc2, 'x', color='lime', markersize=14, markeredgewidth=3)
small_panel(ax, 'WELL-MIXED Fib region inside focal cell\n18S + Fib anchors highlighted', Hl2, L=5)
add_context_inset(ax)
ax = axes[1,1]; im = ax.imshow(cond[ya2:yb2, xa2:xb2], cmap='viridis', vmin=0, vmax=1)
ax.plot(xc2, yc2, 'x', color='lime', markersize=14, markeredgewidth=3)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
small_panel(ax, f'conductance c at marker = {cond[y_mix, x_mix]:.3f}', Hl2, sbcol='black', L=5)
ax = axes[1,2]; im = ax.imshow(rho_smooth[ka, ya2:yb2, xa2:xb2], cmap='hot')
ax.plot(xc2, yc2, 'x', color='lime', markersize=14, markeredgewidth=3)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
small_panel(ax, f'rho_smooth[Fib] at marker: {rho_smooth[ka, y_mix, x_mix]:.4f}', Hl2, L=5)
ax = axes[1,3]; im = ax.imshow(N_eff[ka, ya2:yb2, xa2:xb2], cmap='hot')
ax.plot(xc2, yc2, 'x', color='lime', markersize=14, markeredgewidth=3)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
small_panel(ax, f'N_eff[Fib] at marker: {N_eff[ka, y_mix, x_mix]:.1f}\n(matches local anchor density)', Hl2, L=5)

plt.tight_layout(rect=[0, 0, 1, 0.985])
out = FIG_DIR / 'xenium_skin_db031_bug2-rescaling-mechanism.png'
plt.savefig(out, dpi=130, bbox_inches='tight'); plt.show()
print(f'saved: {out}')

## Finding — closing statement

**FIND-003** (registered 2026-05-24): Per-lineage edge field should be `e_k = ‖∇ max(t_k, 0)‖`, not `‖∇ tanh(K_sharp · s_k)‖`. Clipping `t_k` to `[0, 1]` before the gradient restricts `e_k` to fire only at the perimeter of regions where lineage k is winning. Verified on DB_top100_031: 66% of previously bright pixels drop to exactly 0 under the clip; among 8,174 surviving bright pixels, every single one has a positive `t_Fib` neighbour in a 3×3 window (zero false positives). Real focal-cell Fib boundary preserved.

See `findings_registry.yaml::FIND-003` for the full entry + scope/exceptions, and Diagnostic 3 above for the comparison figure.

**Bug 2** (rescaling by global `2πσ²` over-amplifies disconnected anchors) was identified in this session but **tabled**. See Diagnostic 4 above for the mechanism evidence; candidate fix is local window-integration of `rho_smooth`.